<a href="https://colab.research.google.com/github/dudavsr/arvores_balanceadas_edb2/blob/main/C%C3%B3pia_de_C%C3%B3pia_de_EDB_2_U_2_Arvore_avl_e_flamengo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile arvores.cpp
#include <iostream>
#include <algorithm>
#include <string>
using namespace std;

//Criação árvore AVL
class ArvoreAVL {
private:
    struct No {
        int valor;
        int altura;
        No* esq;
        No* dir;

        //Construtor do nó AVL
        No(int v) {
            valor = v;
            altura = 1;
            esq = nullptr;
            dir = nullptr;
        }};

    No* raiz = nullptr;
    int totalInsercoes = 0;
    int totalRotacoes = 0;
    bool modoTeste = false;

    // Retorna a altura do nó
    int altura(No* no) {
        if (no == nullptr)
            return 0;
        return no->altura;}

    //Calcula FB do nó
    int fatorBalanceamento(No* no) {
        if (no == nullptr)
            return 0;
        //FB = altura esquerda - altura direita
        return altura(no->esq) - altura(no->dir);}

    //Atualiza a altura do nó pós alterações na árvore
    void atualizarAltura(No* no) {
        if (no != nullptr) {
            //Define a nova altura baseada na maior subárvore
            no->altura = 1 + max(altura(no->esq), altura(no->dir));
        }}

    //Rotação simples a direita
    No* rotacaoDireita(No* y) {
        totalRotacoes++;
        No* x = y->esq;
        No* T2 = x->dir;

        //Realiza movimentação dos ponteiros
        x->dir = y;
        y->esq = T2;

        //Recalcula alturas após rotação
        atualizarAltura(y);
        atualizarAltura(x);

        if (!modoTeste)
            cout << "Rotacao direita aplicada em " << y->valor << endl;
        //Novo nó raiz da subárvore
        return x;}

    //Rotação simples a esquerda
    No* rotacaoEsquerda(No* x) {
        totalRotacoes++;
        No* y = x->dir;
        No* T2 = y->esq;

        //Realiza movimentação dos ponteiros
        y->esq = x;
        x->dir = T2;

        //Recalcula alturas após rotação
        atualizarAltura(x);
        atualizarAltura(y);

        if (!modoTeste)
            cout << "Rotacao esquerda aplicada em " << x->valor << endl;
        //Novo nó raiz da subárvore
        return y;}

    //Verifica balanceamento e aplica rotação
    No* balancear(No* no) {
        if (no == nullptr)
            return no;

        atualizarAltura(no);

        //Calcula o FB atual do nó
        int fb = fatorBalanceamento(no);

        //LL
        if (fb > 1 && fatorBalanceamento(no->esq) >= 0) {
            return rotacaoDireita(no);}

        //LR
        if (fb > 1 && fatorBalanceamento(no->esq) < 0) {
            if (!modoTeste)
                cout << "Caso LR detectado em " << no->valor << endl;

            //Primeira rotação no filho esquerdo
            no->esq = rotacaoEsquerda(no->esq);

            //Segunda rotação no nó desbalanceado
            return rotacaoDireita(no);}

        //RR
        if (fb < -1 && fatorBalanceamento(no->dir) <= 0) {
            return rotacaoEsquerda(no);}

        //RL
        if (fb < -1 && fatorBalanceamento(no->dir) > 0) {
            if (!modoTeste)
                cout << "Caso RL detectado em " << no->valor << endl;

            //Primeira rotação no filho direito
            no->dir = rotacaoDireita(no->dir);

            //Segunda rotação no nó desbalanceado
            return rotacaoEsquerda(no);}

        //Retorna nó sem alterações caso já esteja balanceado
        return no;}

    //Inserção recursiva na AVL
    No* inserir(No* no, int valor) {
        if (no == nullptr) {
            //Cria novo nó ao encontrar posição vazia
            return new No(valor);}

        if (valor < no->valor) {
            no->esq = inserir(no->esq, valor);
        } else if (valor > no->valor) {
            no->dir = inserir(no->dir, valor);
        } else {
            //Impede inserção de valores duplicados
            if (!modoTeste)
                cout << "Nao se preocupe, valor repetido ignorado: " << valor << endl;
            return no;}

        //Rebalanceia a árvore após inserção
        return balancear(no);}

    //Encontra o menor valor de uma subárvore
    No* menorValor(No* no) {
        No* atual = no;

        //Percorre sempre para a esquerda
        while (atual != nullptr && atual->esq != nullptr) {
            atual = atual->esq;}

        return atual;}

    //Remove um valor da AVL e equilibra.
    No* remover(No* no, int valor) {
        if (no == nullptr)
            return no;

        if (valor < no->valor) {
            no->esq = remover(no->esq, valor);
        } else if (valor > no->valor) {
            no->dir = remover(no->dir, valor);
        } else {
            //Caso de nó com apenas um filho ou nenhum
            if (no->esq == nullptr || no->dir == nullptr) {
                No* temp;

                if (no->esq != nullptr)
                    temp = no->esq;
                else
                    temp = no->dir;

                //Caso nó folha
                if (temp == nullptr) {
                    temp = no;
                    no = nullptr;
                } else {
                    //Substitui conteúdo do nó removido
                    *no = *temp;}

                delete temp;
            } else {
                //Busca sucessor imediato para substituição
                No* temp = menorValor(no->dir);

                no->valor = temp->valor;

                //Remove o sucessor da posição original
                no->dir = remover(no->dir, temp->valor);
            }}

        if (no == nullptr)
            return no;

        //Rebalanceia árvore após remoção
        return balancear(no);}

    //Busca recursiva de tal valor
    bool buscar(No* no, int valor) {
        if (no == nullptr)
            return false;

        if (valor == no->valor)
            return true;

        //Busca na subárvore esquerda
        if (valor < no->valor)
            return buscar(no->esq, valor);

        //Busca na subárvore direita
        return buscar(no->dir, valor);}

    //Representação visual
    void imprimir(No* no, string espaco, bool direita) {
        if (no == nullptr)
            return;

        espaco += "     ";

        //Imprime primeiro a subárvore direita
        imprimir(no->dir, espaco, true);

        cout << espaco.substr(5);

        if (direita)
            cout << "┌──";
        else
            cout << "└──";

        //Exibe valor, altura e FB do nó
        cout << no->valor
             << "(h=" << no->altura
             << ",fb=" << fatorBalanceamento(no)
             << ")" << endl;

        //Imprime depois a subárvore esquerda
        imprimir(no->esq, espaco, false);}

    //Liberação de memória
    void destruir(No* no) {
        if (no == nullptr)
            return;

        //Percorre árvore em pós-ordem
        destruir(no->esq);
        destruir(no->dir);
        //Libera memória do nó atual
        delete no;}

public:
    ~ArvoreAVL() {
        //Libera toda memória alocada dinamicamente
        destruir(raiz);}

    void inserir(int valor) {
        totalInsercoes++;
        //Atualiza a raiz após possível rebalanceamento
        raiz = inserir(raiz, valor);}

    void remover(int valor) {
        //Atualiza a raiz após possível rebalanceamento
        raiz = remover(raiz, valor);}

    //Busca pública de um valor na AVL
    void buscar(int valor) {
        if (buscar(raiz, valor)) {
            cout << "Valor " << valor << " encontrado na AVL." << endl;
        } else {
            cout << "Valor " << valor << " nao encontrado na AVL." << endl;
        }}

    void imprimir() {
        if (raiz == nullptr) {
            cout << "Arvore AVL vazia." << endl;
            return;}

        cout << "\nArvore AVL:\n";
        imprimir(raiz, "", true);}

    void ativarModoTeste() {
        modoTeste = true;
    }

    int getInsercoes() {
        return totalInsercoes;
    }

    int getRotacoes() {
        return totalRotacoes;
    }
};

//Criação árvore Rubro-Negra
class ArvoreRubroNegra {
private:
    enum Cor {
        VERMELHO,
        PRETO};

    struct No {
        int valor;
        Cor cor;
        No* esq;
        No* dir;
        No* pai;

        //Construtor do nó Rubro-Negro
        No(int v = 0) {
            valor = v;
            cor = VERMELHO;
            esq = nullptr;
            dir = nullptr;
            pai = nullptr;}
    };

    No* NIL;
    No* raiz;
    int totalInsercoes = 0;
    int totalRotacoes = 0;
    bool modoTeste = false;

    //Rotação a esquerda na Rubro-Negra
    void rotacaoEsquerda(No* x) {
        totalRotacoes++;
        No* y = x->dir;
        x->dir = y->esq;

        //Atualiza pai da subárvore movida
        if (y->esq != NIL) {
            y->esq->pai = x;}

        y->pai = x->pai;

        //Atualiza referência da raiz se necessário
        if (x->pai == NIL) {
            raiz = y;
        } else if (x == x->pai->esq) {
            x->pai->esq = y;
        } else {
            x->pai->dir = y;}

        //Finaliza rotação
        y->esq = x;
        x->pai = y;

        if (!modoTeste)
            cout << "Rotacao esquerda aplicada em " << x->valor << endl;}

    //Rotação a direita na Rubro-Negra
    void rotacaoDireita(No* y) {
        totalRotacoes++;
        No* x = y->esq;
        y->esq = x->dir;

        //Atualiza pai da subárvore movida
        if (x->dir != NIL) {
            x->dir->pai = y;}

        x->pai = y->pai;

        //Atualiza referência da raiz se necessário
        if (y->pai == NIL) {
            raiz = x;
        } else if (y == y->pai->dir) {
            y->pai->dir = x;
        } else {
            y->pai->esq = x;}

        //Finaliza rotação
        x->dir = y;
        y->pai = x;

        if (!modoTeste)
            cout << "Rotacao direita aplicada em " << y->valor << endl;}

    //Corrigindo possível erro pós inserção
    void corrigirInsercao(No* z) {

        //Enquanto existir conflito vermelho-vermelho
        while (z->pai->cor == VERMELHO) {
            if (z->pai == z->pai->pai->esq) {
                No* tio = z->pai->pai->dir;
                //Caso de recoloração
                if (tio->cor == VERMELHO) {
                    if (!modoTeste)
                        cout << "Recoloracao na insercao envolvendo " << z->valor << endl;

                    z->pai->cor = PRETO;
                    tio->cor = PRETO;
                    z->pai->pai->cor = VERMELHO;
                    z = z->pai->pai;
                } else {
                    //Caso LR
                    if (z == z->pai->dir) {
                        z = z->pai;
                        rotacaoEsquerda(z);}

                    z->pai->cor = PRETO;
                    z->pai->pai->cor = VERMELHO;
                    rotacaoDireita(z->pai->pai);}
            } else {
                No* tio = z->pai->pai->esq;

                //Caso de recoloração
                if (tio->cor == VERMELHO) {
                    if (!modoTeste)
                        cout << "Recoloracao na insercao envolvendo " << z->valor << endl;

                    z->pai->cor = PRETO;
                    tio->cor = PRETO;
                    z->pai->pai->cor = VERMELHO;
                    z = z->pai->pai;
                } else {
                    //Caso RL
                    if (z == z->pai->esq) {
                        z = z->pai;
                        rotacaoDireita(z);}

                    z->pai->cor = PRETO;
                    z->pai->pai->cor = VERMELHO;
                    rotacaoEsquerda(z->pai->pai);
                }}}
        //Raiz sempre deve ser preta
        raiz->cor = PRETO;}

    //Substitui uma subárvore por outra durante remoção
    void transplantar(No* u, No* v) {

        //Atualiza raiz caso necessário
        if (u->pai == NIL) {
            raiz = v;
        } else if (u == u->pai->esq) {
            u->pai->esq = v;
        } else {
            u->pai->dir = v;}
        //Atualiza referência do pai
        v->pai = u->pai;}

    // Menor nó da subárvore
    No* minimo(No* no) {
        //Percorre sempre para a esquerda
        while (no->esq != NIL) {
            no = no->esq;}

        return no;}

    // Corrigindo possível erro pós remoção
    void corrigirRemocao(No* x) {

        //Executa enquanto existir duplo-preto
        while (x != raiz && x->cor == PRETO) {
            if (x == x->pai->esq) {
                No* w = x->pai->dir;

                //Irmão vermelho
                if (w->cor == VERMELHO) {
                    w->cor = PRETO;
                    x->pai->cor = VERMELHO;
                    rotacaoEsquerda(x->pai);
                    w = x->pai->dir;}

                //Irmão com filhos pretos
                if (w->esq->cor == PRETO && w->dir->cor == PRETO) {
                    w->cor = VERMELHO;
                    x = x->pai;
                } else {

                    //Rotação preparatória
                    if (w->dir->cor == PRETO) {
                        w->esq->cor = PRETO;
                        w->cor = VERMELHO;
                        rotacaoDireita(w);
                        w = x->pai->dir;}

                    //Rotação final de correção
                    w->cor = x->pai->cor;
                    x->pai->cor = PRETO;
                    w->dir->cor = PRETO;
                    rotacaoEsquerda(x->pai);
                    x = raiz;}
            } else {
                No* w = x->pai->esq;

                //Irmão vermelho
                if (w->cor == VERMELHO) {
                    w->cor = PRETO;
                    x->pai->cor = VERMELHO;
                    rotacaoDireita(x->pai);
                    w = x->pai->esq;}

                //Irmão com filhos pretos
                if (w->dir->cor == PRETO && w->esq->cor == PRETO) {
                    w->cor = VERMELHO;
                    x = x->pai;
                } else {

                    //Rotação preparatória
                    if (w->esq->cor == PRETO) {
                        w->dir->cor = PRETO;
                        w->cor = VERMELHO;
                        rotacaoEsquerda(w);
                        w = x->pai->esq;}

                    //Rotação final de correção
                    w->cor = x->pai->cor;
                    x->pai->cor = PRETO;
                    w->esq->cor = PRETO;
                    rotacaoDireita(x->pai);
                    x = raiz;
                    }}}
        //Garante nó final preto
        x->cor = PRETO;}

    //Busca um nó específico
    No* buscarNo(No* no, int valor) {
        //Retorna NIL caso valor não exista
        if (no == NIL || valor == no->valor) {
            return no;}
        //Busca recursiva na esquerda
        if (valor < no->valor) {
            return buscarNo(no->esq, valor);}
        //Busca recursiva na direita
        return buscarNo(no->dir, valor);}

    //Impressão da arvore com cores de nós
    void imprimir(No* no, string espaco, bool direita) {
        if (no == NIL)
            return;

        espaco += "     ";

        //Imprime primeiro a subárvore direita
        imprimir(no->dir, espaco, true);

        cout << espaco.substr(5);

        if (direita)
            cout << "┌──";
        else
            cout << "└──";

        cout << no->valor << "(";

        //Exibe cor do nó
        if (no->cor == VERMELHO)
            cout << "V";
        else
            cout << "P";

        cout << ")" << endl;

        //Imprime depois a subárvore esquerda
        imprimir(no->esq, espaco, false);}

    //Liberação de memória
    void destruir(No* no) {
        if (no == NIL)
            return;

        //Percorre árvore em pós-ordem
        destruir(no->esq);
        destruir(no->dir);

        //Libera memória do nó atual
        delete no;}

public:
    ArvoreRubroNegra() {
        //Inicializa nó sentinela NIL
        NIL = new No();
        NIL->cor = PRETO;
        NIL->esq = NIL;
        NIL->dir = NIL;
        NIL->pai = NIL;

        //Árvore inicia vazia
        raiz = NIL;}

    ~ArvoreRubroNegra() {
        //Libera todos os nós da árvore
        destruir(raiz);

        //Libera nó sentinela
        delete NIL;}

    void inserir(int valor) {

        //Impede inserção duplicada
        if (buscarNo(raiz, valor) != NIL) {
            if (!modoTeste)
                cout << "Valor repetido ignorado: " << valor << endl;
            return;}

        totalInsercoes++;
        No* z = new No(valor);

        //Inicializa filhos como NIL
        z->esq = NIL;
        z->dir = NIL;

        No* y = NIL;
        No* x = raiz;

        //Busca posição correta de inserção
        while (x != NIL) {
            y = x;

            if (z->valor < x->valor)
                x = x->esq;
            else
                x = x->dir;}

        z->pai = y;

        //Caso árvore vazia
        if (y == NIL) {
            raiz = z;
        } else if (z->valor < y->valor) {
            y->esq = z;
        } else {
            y->dir = z;}

        //Novo nó sempre inicia vermelho
        z->cor = VERMELHO;

        //Corrige possíveis violações
        corrigirInsercao(z);}

    void remover(int valor) {
        No* z = buscarNo(raiz, valor);

        //Verifica se valor existe
        if (z == NIL) {
            cout << "Valor " << valor << " nao encontrado para remocao, desculpa." << endl;
            return;}

        No* y = z;
        No* x;

        //Armazena cor original para verificar necessidade de correção
        Cor corOriginal = y->cor;

        //Caso sem filho esquerdo
        if (z->esq == NIL) {
            x = z->dir;
            transplantar(z, z->dir);

        //Caso sem filho direito
        } else if (z->dir == NIL) {
            x = z->esq;
            transplantar(z, z->esq);

        //Caso com dois filhos
        } else {
            //Busca sucessor imediato
            y = minimo(z->dir);
            corOriginal = y->cor;
            x = y->dir;

            if (y->pai == z) {
                x->pai = y;
            } else {
                //Remove sucessor da posição original
                transplantar(y, y->dir);
                y->dir = z->dir;
                y->dir->pai = y;}

            //Substitui nó removido pelo sucessor
            transplantar(z, y);
            y->esq = z->esq;
            y->esq->pai = y;

            //Mantém cor original do nó removido
            y->cor = z->cor;}

        delete z;

        //Corrige possíveis violações da Rubro-Negra
        if (corOriginal == PRETO) {
            corrigirRemocao(x);
        }}

    void buscar(int valor) {
        if (buscarNo(raiz, valor) != NIL) {
            cout << "Valor " << valor << " encontrado na Rubro-Negra!" << endl;
        } else {
            cout << "Valor " << valor << " nao encontrado na Rubro-Negra, desculpa." << endl;
        }}

    void imprimir() {

        //Verifica se árvore está vazia
        if (raiz == NIL) {
            cout << "Arvore Rubro-Negra vazia." << endl;
            return;}

        cout << "\nArvore Rubro-Negra:\n";

        //Inicia impressão pela raiz
        imprimir(raiz, "", true);
    }

    void ativarModoTeste() {
        modoTeste = true;
    }

    int getInsercoes() {
        return totalInsercoes;
    }

    int getRotacoes() {
        return totalRotacoes;
    }
};

//Teste com inserção de 20000 números nas árvores para comparação
void testeAutomatico() {

    ArvoreAVL avl;
    ArvoreRubroNegra rn;

    avl.ativarModoTeste();
    rn.ativarModoTeste();

    cout << "\n===== TESTE AUTOMATICO =====\n";

    //Silencia prints internos
    cout.setstate(ios_base::failbit);

    //Insercao de 20000 valores
    for (int i = 1; i <= 20000; i++) {

        avl.inserir(i);
        rn.inserir(i);
    }
    //Reativa cout
    cout.clear();

    cout << "\nAVL\n";
    cout << "Insercoes: " << avl.getInsercoes() << endl;
    cout << "Rotacoes: " << avl.getRotacoes() << endl;

    cout << "\nRubro-Negra\n";
    cout << "Insercoes: " << rn.getInsercoes() << endl;
    cout << "Rotacoes: " << rn.getRotacoes() << endl;

    cout << "\n============================\n";
}

// Menu da AVL, com as possibilidade de ações pedidas pela tarefa
// ações implementadas antes e utilizadas aqui para ficar mais organizado o código

void menuAVL() {
    ArvoreAVL arvore;
    int opcao;
    int valor;

    do {
        cout << "\n   MENU AVL   \n";
        cout << "1 - Inserir\n";
        cout << "2 - Remover\n";
        cout << "3 - Buscar\n";
        cout << "4 - Imprimir arvore\n";
        cout << "0 - Sair\n";
        cout << "Escolha: ";
        cin >> opcao;

        switch (opcao) {
            case 1:
                cout << "Valor para inserir: ";
                cin >> valor;
                arvore.inserir(valor);
                break;

            case 2:
                cout << "Valor para remover: ";
                cin >> valor;
                arvore.remover(valor);
                break;

            case 3:
                cout << "Valor para buscar: ";
                cin >> valor;
                arvore.buscar(valor);
                break;

            case 4:
                arvore.imprimir();
                break;

            case 0:
                cout << "Saindo...\n";
                break;

            default:
                cout << "Opcao invalida.\n";}

    } while (opcao != 0);}

// Menu da Rubro-Negra, com as possibilidade de ações pedidas pela tarefa
// ações implementadas antes e utilizadas aqui para ficar mais organizado o código

void menuRubroNegra() {
    ArvoreRubroNegra arvore;
    int opcao;
    int valor;

    do {
        cout << "\n   MENU RUBRO-NEGRA   \n";
        cout << "1 - Inserir\n";
        cout << "2 - Remover\n";
        cout << "3 - Buscar\n";
        cout << "4 - Imprimir arvore\n";
        cout << "0 - Sair\n";
        cout << "Escolha: ";
        cin >> opcao;

        switch (opcao) {
            case 1:
                cout << "Valor para inserir: ";
                cin >> valor;
                arvore.inserir(valor);
                break;

            case 2:
                cout << "Valor para remover: ";
                cin >> valor;
                arvore.remover(valor);
                break;

            case 3:
                cout << "Valor para buscar: ";
                cin >> valor;
                arvore.buscar(valor);
                break;

            case 4:
                arvore.imprimir();
                break;

            case 0:
                cout << "Saindo...\n";
                break;

            default:
                cout << "Opcao invalida.\n";
        }}
         while (opcao != 0);}

//Função Main: escolha do tipo de arvóre e a partir dai redireciona
int main() {

    int escolha;

    cout << "\n";
    cout << "Sistema de Arvores Balanceadas\n";
    cout << "1 - Arvore AVL\n";
    cout << "2 - Arvore Rubro-Negra\n";
    cout << "3 - Executar teste automatico\n";
    cout << "Por favor, escolha o tipo de arvore, digite o numero: ";
    cin >> escolha;

    if (escolha == 1) {
        menuAVL();

    } else if (escolha == 2) {
        menuRubroNegra();

    } else if (escolha == 3) {
        testeAutomatico();

    } else {
        cout << "Opcao invalida. Encerrando programa.\n";
    }

    return 0;}

Overwriting arvores.cpp


In [ ]:
!g++ arvores.cpp -o arvores

In [ ]:
!./arvores


Sistema de Arvores Balanceadas
1 - Arvore AVL
2 - Arvore Rubro-Negra
3 - Executar teste automatico
Por favor, escolha o tipo de arvore, digite o numero: 3

===== TESTE AUTOMATICO =====

AVL
Insercoes: 20001
Rotacoes: 19986

Rubro-Negra
Insercoes: 20001
Rotacoes: 19975

